## Setup

In [2]:
import optuna
from optuna.samplers import TPESampler
from optuna.trial import Trial
from optuna.study import create_study
import diff_cont
import lambda_cont
import libs.agent_infra as ai
import os

import json
import datetime
import itertools
import constants as Cs
import concurrent.futures
import numpy as np
from concurrent.futures import ProcessPoolExecutor

SEEDS = [101, 102, 103]
TEST_EVAL_EPS = 6

/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-08 22:56:49.184903: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-08 22:56:50.543079: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


## Add Novelty Diff

In [5]:
def objective(trial:Trial):
    env = Cs.ENIVROMENTS["lunarlander"]()
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 0.5, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    fit_w = trial.suggest_float("start_fit_w", 0.2, 1, step=0.1)
    decay = trial.suggest_float("decay", 0.5, 5, step=0.5)

    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, 
                env="lunarlander",
                container="add_novelty",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                fitness_weight=fit_w,
                decay=decay,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]
    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler(n_startup_trials=25)
study = create_study(direction="maximize", sampler=sampler, study_name="diff_add_novelty_exp", storage="sqlite:///Data/optuna/lunarlander/add_novelty/diff.db", load_if_exists=True)
study.optimize(objective, n_trials=180, n_jobs=5)
print(study.best_trials)

[I 2026-06-01 01:10:05,589] Using an existing study with name 'diff_add_novelty_exp' instead of creating a new one.


   	                        fitness                        	                            novelty                             
   	-------------------------------------------------------	----------------------------------------------------------------
gen	avg    	gen	max     	min     	std    	avg     	gen	max     	min      	std     
0  	-338.14	0  	-87.8815	-561.809	141.658	0.242896	0  	0.897891	0.0229698	0.195207
   	                            fitness                            	                            novelty                            
   	---------------------------------------------------------------	---------------------------------------------------------------
gen	avg     	gen	max     	min     	std    	avg     	gen	max     	min       	std    
0  	-405.342	0  	-87.4639	-712.778	174.504	0.220417	0  	0.821986	0.00233417	0.16461
   	                            fitness                            	                            novelty                             
   	---------------

[I 2026-06-01 01:47:01,921] Trial 6 finished with value: -65.09519064088458 and parameters: {'lambda': 40, 'mutation_rate': 0.33, 'cross_rate': 0.8, 'start_fit_w': 0.7, 'decay': 4.5}. Best is trial 6 with value: -65.09519064088458.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A clas

14 	-358.373	14 	-11.5433	-678.318	177.643	0.375996	14 	0.711172	0.00163325 	0.191068
14 	-385.46 	14 	3.05532 	-672.309	180.85 	0.342853	14 	0.704644	0.00375791	0.184085
   	                            fitness                            	                            novelty                            
   	---------------------------------------------------------------	---------------------------------------------------------------
gen	avg     	gen	max     	min     	std    	avg     	gen	max     	min      	std    
0  	-363.603	0  	-103.371	-552.597	138.888	0.377583	0  	0.826911	0.0379985	0.24305
   	                            fitness                            	                            novelty                             
   	---------------------------------------------------------------	----------------------------------------------------------------
gen	avg     	gen	max     	min     	std    	avg     	gen	max     	min       	std     
0  	-385.433	0  	-88.0261	-712.513	175.138	0.443

[I 2026-06-01 01:59:53,327] Trial 9 finished with value: 0.21203780355286975 and parameters: {'lambda': 50, 'mutation_rate': 0.46, 'cross_rate': 0.8, 'start_fit_w': 0.9000000000000001, 'decay': 3.0}. Best is trial 9 with value: 0.21203780355286975.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: Runti

...

1  	-396.649	1  	-12.1332	-726.101	175.252	0.302664	1  	0.648042	0.0313669	0.156513
1  	-387.278	1  	-90.6226	-712.513	170.854	0.298563	1  	0.687205	0.00483602	0.158813
2  	-338.271	2  	-54.171 	-561.185	146.86 	0.293725	2  	0.65431 	0.00337107	0.163237
6  	-367.875	6  	-15.5604	-712.513	190.656	0.301079	6  	0.674765	0.00163374	0.158615
1  	-396.649	1  	-12.1332	-726.101	175.252	0.302664	1  	0.648042	0.0313669	0.156513
6  	-404.034	6  	-30.9188	-658.757	161.696	0.278008	6  	0.601408	0.0113344  	0.138141
2  	-343.453	2  	-76.6882	-573.399	146.668	0.311121	2  	0.666775	0.0212976	0.157254
7  	-345.761	7  	-71.4527	-617.508	137.792	0.323454	7  	0.732204	0.0637853 	0.157753
2  	-404.023	2  	-100.181	-712.513	154.419	0.301114	2  	0.753916	0.00458311	0.166085
2  	-361.885	2  	2.75909 	-687.54 	153.3  	0.308932	2  	0.646955	0.0391452  	0.146422
2  	-343.453	2  	-76.6882	-573.399	146.668	0.311121	2  	0.666775	0.0212976	0.157254
2  	-401.797	2  	-74.2771	-712.513	164.376	0.29774 	2  	0.771086	0.

[I 2026-06-02 02:17:41,736] Trial 179 finished with value: 72.50437088943481 and parameters: {'lambda': 60, 'mutation_rate': 0.07, 'cross_rate': 0.3, 'start_fit_w': 0.5, 'decay': 2.5}. Best is trial 85 with value: 82.85210520757455.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class n

   	                            fitness                            	                            novelty                             
   	---------------------------------------------------------------	----------------------------------------------------------------
gen	avg     	gen	max     	min     	std    	avg     	gen	max     	min     	std     
0  	-357.539	0  	-91.1992	-596.391	134.434	0.310508	0  	0.795728	0.067223	0.167483
   	                            fitness                            	                            novelty                             
   	---------------------------------------------------------------	----------------------------------------------------------------
gen	avg     	gen	max     	min     	std    	avg     	gen	max     	min     	std     
0  	-403.019	0  	-35.6292	-630.771	162.684	0.307212	0  	0.761093	0.010928	0.188368
   	                            fitness                            	                            novelty                             
   

[I 2026-06-02 02:32:31,054] Trial 180 finished with value: 72.50437088943481 and parameters: {'lambda': 60, 'mutation_rate': 0.07, 'cross_rate': 0.3, 'start_fit_w': 0.5, 'decay': 2.0}. Best is trial 85 with value: 82.85210520757455.
[I 2026-06-02 02:32:56,164] Trial 181 finished with value: 82.85210520757455 and parameters: {'lambda': 60, 'mutation_rate': 0.09, 'cross_rate': 0.3, 'start_fit_w': 0.5, 'decay': 2.0}. Best is trial 85 with value: 82.85210520757455.
[I 2026-06-02 02:33:08,641] Trial 182 finished with value: 82.85210520757455 and parameters: {'lambda': 60, 'mutation_rate': 0.09, 'cross_rate': 0.3, 'start_fit_w': 0.5, 'decay': 2.0}. Best is trial 85 with value: 82.85210520757455.
[I 2026-06-02 02:33:33,282] Trial 183 finished with value: 82.85210520757455 and parameters: {'lambda': 60, 'mutation_rate': 0.09, 'cross_rate': 0.3, 'start_fit_w': 0.5, 'decay': 2.0}. Best is trial 85 with value: 82.85210520757455.
[I 2026-06-02 02:35:37,440] Trial 184 finished with value: 82.852105

[FrozenTrial(number=85, state=<TrialState.COMPLETE: 1>, values=[82.85210520757455], datetime_start=datetime.datetime(2026, 6, 1, 12, 20, 38, 681537), datetime_complete=datetime.datetime(2026, 6, 1, 13, 1, 7, 636831), params={'lambda': 60, 'mutation_rate': 0.09, 'cross_rate': 0.3, 'start_fit_w': 0.5, 'decay': 1.0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'lambda': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mutation_rate': FloatDistribution(high=0.5, log=False, low=0.0, step=0.01), 'cross_rate': FloatDistribution(high=1.0, log=False, low=0.3, step=0.1), 'start_fit_w': FloatDistribution(high=1.0, log=False, low=0.2, step=0.1), 'decay': FloatDistribution(high=5.0, log=False, low=0.5, step=0.5)}, trial_id=266, value=None), FrozenTrial(number=101, state=<TrialState.COMPLETE: 1>, values=[82.85210520757455], datetime_start=datetime.datetime(2026, 6, 1, 14, 41, 47, 518944), datetime_complete=datetime.datetime(2026, 6, 1, 15, 27, 6, 438380), params={'lambd

## Fit_Archiving

In [3]:
def objective(trial:Trial):
    environment = "lunarlander"
    env = Cs.ENIVROMENTS[environment]()
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 1, step=0.1)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    archiving = trial.suggest_int("archiving_period", 2, 5)
    archive_batch = trial.suggest_int("archive_batch", 1, 5)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, env=environment,
                container="fit_archiving",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                archiving_period=archiving,
                archive_batch=archive_batch,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler(n_startup_trials=20)
diff_fita_study = create_study(direction="maximize", study_name="diff_fit_archiving_reloaded", sampler=sampler, storage="sqlite:///Data/optuna/lunarlander/fit_archiving/diff.db", load_if_exists=True)
diff_fita_study.optimize(objective, n_trials=150, n_jobs=5)
print(diff_fita_study.best_trials)

[I 2026-06-02 15:47:00,799] Using an existing study with name 'diff_fit_archiving_reloaded' instead of creating a new one.


gen	avg     	min     	max     	std    
0  	-262.287	-471.163	-117.798	124.336
gen	avg     	min     	max    	std    
0  	-300.311	-490.694	-56.647	117.984
gen	avg   	min     	max    	std    
0  	-302.3	-572.813	8.05399	164.042
gen	avg     	min     	max     	std    
0  	-251.112	-550.176	-42.3605	125.151
gen	avg     	min     	max     	std    
0  	-309.833	-524.709	-96.2772	134.192
gen	avg     	min     	max     	std    
0  	-297.735	-550.176	-103.281	141.153
1  	-234.636	-484.746	-56.647	118.735
gen	avg     	min     	max     	std    
0  	-278.073	-489.993	-70.8479	114.727
1  	-234.175	-452.994	-55.1376	124.195
gen	avg     	min     	max     	std    
0  	-283.735	-572.813	-67.2347	160.231
gen	avg     	min     	max     	std    
0  	-318.366	-545.777	-67.5599	140.116
gen	avg     	min     	max     	std    
0  	-304.633	-512.611	-74.5287	142.464
1  	-226.715	-550.176	-103.281	107.617
1  	-254.569	-523.548	8.05399	146.923
1  	-281.611	-492.608	-70.4645	134.96 
1  	-229.793	-493.213	-37.2978	114.

[I 2026-06-02 16:37:13,843] Trial 8 finished with value: -67.41815377168389 and parameters: {'lambda': 40, 'mutation_rate': 0.2, 'cross_rate': 0.5, 'archiving_period': 3, 'archive_batch': 1}. Best is trial 8 with value: -67.41815377168389.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A clas

13 	-153.982	-486.349	-55.8383	99.7477
14 	-94.5335	-233.634	-48.9372	22.1082
12 	-124.906	-325.617	-27.2369	64.4004
14 	-143.142	-346.915	-2.49628	76.1437
13 	-166.69 	-470.077	43.621  	99.2915


[I 2026-06-02 16:38:48,190] Trial 7 finished with value: -21.000206077986448 and parameters: {'lambda': 40, 'mutation_rate': 0.2, 'cross_rate': 0.9000000000000001, 'archiving_period': 2, 'archive_batch': 3}. Best is trial 7 with value: -21.000206077986448.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: Runti

...

8  	-160.799	-414.577	30.5769	135.585
2  	-230.571	-472.303	-55.8383	136.127
9  	-131.209	-458.761	-55.8383	95.9765
8  	-112.147	-377.293	80.17   	96.4192
2  	-185.447	-482.219	-16.7504	114.091
2  	-230.571	-472.303	-55.8383	136.127
2  	-285.088	-572.441	60.2876	149.015
2  	-185.447	-482.219	-16.7504	114.091
3  	-199.38 	-453.451	-55.8383	117.395
2  	-285.088	-572.441	60.2876	149.015
10 	-122.814	-429.626	-55.8383	75.6615
9  	-116.254	-414.577	30.5769	103.635
9  	-101.416	-377.293	80.17   	82.5138
3  	-199.38 	-453.451	-55.8383	117.395
3  	-164.65 	-476.982	-16.7504	103.884
3  	-241.641	-523.548	26.4315	142.731
3  	-164.65 	-476.982	-16.7504	103.884
4  	-181.429	-453.451	-55.8383	112.931
3  	-241.641	-523.548	26.4315	142.731
11 	-119.248	-429.626	-44.6016	72.0964
10 	-95.8739	-324.635	30.5769	76.7787
10 	-100.662	-360.008	80.17   	83.6913
4  	-181.429	-453.451	-55.8383	112.931
4  	-153.285	-476.982	20.2796 	113.093
4  	-195.863	-523.548	26.4315	140.097
4  	-153.285	-476.982	20.2796 	11

[I 2026-06-03 16:54:07,550] Trial 152 finished with value: 45.404550512338965 and parameters: {'lambda': 70, 'mutation_rate': 0.0, 'cross_rate': 0.3, 'archiving_period': 2, 'archive_batch': 2}. Best is trial 50 with value: 80.34870366377301.


14 	-118.947	-310.331	61.6795 	78.572 
14 	-94.1799	-324.635	42.5214	88.059 
14 	-118.947	-310.331	61.6795 	78.572 
14 	-94.1799	-324.635	42.5214	88.059 


[I 2026-06-03 17:10:43,854] Trial 153 finished with value: 45.404550512338965 and parameters: {'lambda': 70, 'mutation_rate': 0.0, 'cross_rate': 0.3, 'archiving_period': 2, 'archive_batch': 2}. Best is trial 50 with value: 80.34870366377301.
[I 2026-06-03 17:10:52,620] Trial 154 finished with value: 45.404550512338965 and parameters: {'lambda': 70, 'mutation_rate': 0.0, 'cross_rate': 0.3, 'archiving_period': 2, 'archive_batch': 2}. Best is trial 50 with value: 80.34870366377301.
Process ForkProcess-157:
Process ForkProcess-159:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args

KeyboardInterrupt: 

## Novlety Limit Archiving

In [ ]:
def objective(trial:Trial):
    environment = "lunarlander"
    env = Cs.ENIVROMENTS[environment]()
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 1, step=0.1)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    archiving = trial.suggest_int("archiving_period", 2, 5)
    archive_batch = trial.suggest_int("archive_batch", 1, 5)
    limit = trial.suggest_float("limit", -200, -50, step=10)

    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, env=environment,
                container="novelty_limit",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                archiving_period=archiving,
                archive_batch=archive_batch,
                limit=limit,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler(n_startup_trials=20)
diff_fita_study = create_study(direction="maximize", study_name="diff_novelty_limit", sampler=sampler, storage="sqlite:///Data/optuna/lunarlander/novelty_limit/diff.db", load_if_exists=True)
diff_fita_study.optimize(objective, n_trials=170, n_jobs=5)
print(diff_fita_study.best_trials)

OperationalError: (sqlite3.OperationalError) unable to open database file
(Background on this error at: https://sqlalche.me/e/20/e3q8)

## Novelty Archiving

In [ ]:
def objective(trial:Trial):
    environment = "lunarlander"
    env = Cs.ENIVROMENTS[environment]()
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 1, step=0.1)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    archiving = trial.suggest_int("archiving_period", 2, 5)
    archive_batch = trial.suggest_int("archive_batch", 1, 5)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, env=environment,
                container="novelty_archiving",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                archiving_period=archiving,
                archive_batch=archive_batch,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler(n_startup_trials=20)
diff_fita_study = create_study(direction="maximize", study_name="diff_novelty_archiving", sampler=sampler, storage="sqlite:///Data/optuna/lunarlander/novelty_archiving/diff.db", load_if_exists=True)
diff_fita_study.optimize(objective, n_trials=150, n_jobs=5)
print(diff_fita_study.best_trials)

[I 2026-06-04 01:03:50,987] Using an existing study with name 'diff_novelty_archiving' instead of creating a new one.


   	                            fitness                            	                    novelty                     
   	---------------------------------------------------------------	------------------------------------------------
gen	avg     	gen	max     	min     	std    	avg    	gen	max    	min     	std     
0  	-341.901	0  	-111.557	-606.731	161.466	1.39071	0  	4.41253	0.981356	0.840566
   	                            fitness                            	                novelty                
   	---------------------------------------------------------------	---------------------------------------
gen	avg     	gen	max     	min     	std    	avg    	gen	max    	min    	std    
0  	-365.884	0  	-89.9825	-587.226	156.703	1.92353	0  	6.45811	1.11371	1.37082
   	                            fitness                            	                   novelty                    
   	---------------------------------------------------------------	----------------------------------------------


[I 2026-06-04 02:15:14,149] Trial 17 finished with value: -108.94013012164096 and parameters: {'lambda': 50, 'mutation_rate': 0.1, 'cross_rate': 0.3, 'archiving_period': 3, 'archive_batch': 4}. Best is trial 17 with value: -108.94013012164096.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning

   	                            fitness                            	                novelty                
   	---------------------------------------------------------------	---------------------------------------
gen	avg     	gen	max     	min     	std    	avg   	gen	max    	min    	std    
0  	-344.445	0  	-125.803	-557.658	159.866	1.8445	0  	6.69578	1.07569	1.32532
   	                           fitness                            	                novelty                
   	--------------------------------------------------------------	---------------------------------------
gen	avg     	gen	max     	min     	std   	avg    	gen	max    	min    	std    
0  	-363.442	0  	-84.6162	-712.513	177.43	1.67928	0  	4.31197	1.06492	0.94355
   	                        fitness                        	                novelty                
   	-------------------------------------------------------	---------------------------------------
gen	avg     	gen	max    	min     	std    	avg    	gen	max 

[I 2026-06-04 02:26:09,591] Trial 15 finished with value: -14.273235031589271 and parameters: {'lambda': 50, 'mutation_rate': 0.8, 'cross_rate': 1.0, 'archiving_period': 4, 'archive_batch': 3}. Best is trial 15 with value: -14.273235031589271.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessTrue' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarnin

...

[I 2026-06-04 22:20:19,558] Trial 162 finished with value: -14.273235031589271 and parameters: {'lambda': 50, 'mutation_rate': 0.8, 'cross_rate': 1.0, 'archiving_period': 3, 'archive_batch': 2}. Best is trial 15 with value: -14.273235031589271.
[I 2026-06-04 22:20:27,438] Trial 163 finished with value: -14.273235031589271 and parameters: {'lambda': 50, 'mutation_rate': 0.8, 'cross_rate': 1.0, 'archiving_period': 3, 'archive_batch': 2}. Best is trial 15 with value: -14.273235031589271.


14 	-446.925	14 	-91.7997	-633.659	189.688	1.6463  	14 	4.26375 	1.39038 	0.441994 
13 	-337.437	13 	-152.62 	-682.89 	220.061	2.67834 	13 	6.87908	2.10821 	1.28334 
14 	-349.022	14 	-152.62 	-682.89 	224.561	2.28821 	14 	7.476  	1.51072 	1.62245 


[I 2026-06-04 22:28:43,054] Trial 164 finished with value: -14.273235031589271 and parameters: {'lambda': 50, 'mutation_rate': 0.8, 'cross_rate': 1.0, 'archiving_period': 3, 'archive_batch': 2}. Best is trial 15 with value: -14.273235031589271.
Process ForkProcess-197:
Process ForkProcess-198:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/concu

KeyboardInterrupt: 

[W 2026-06-05 21:34:12,559] Trial 75 failed with parameters: {'lambda': 50, 'mutation_rate': 0.7000000000000001, 'cross_rate': 1.0, 'archiving_period': 3, 'archive_batch': 4} because of the following error: BrokenProcessPool('A process in the process pool was terminated abruptly while the future was running or pending.').
Traceback (most recent call last):
  File "/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_246433/1057210813.py", line 22, in objective
    pops = [f.result()[1] for f in futures]
  File "/tmp/ipykernel_246433/1057210813.py", line 22, in <listcomp>
    pops = [f.result()[1] for f in futures]
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/concurrent/futures/_base.py", line 458, in result
    return self.__get_result()
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/concurrent/futures/_base.py", line 403, in __g